In [4]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFilter, ImageFont
import gradio as gr
from google.colab import drive
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from datetime import datetime

# Install dependencies
try:
    import gradio
except ImportError:
    print("📦 Installing dependencies...")
    !pip install gradio==4.19.0 plotly -q
    print("✅ Dependencies installed. NOTE: You may need to restart the runtime if Gradio fails to launch.")

try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully")
except:
    print("⚠️ Could not mount Google Drive - continuing without it")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully


In [5]:
BASE_FOLDER = "/content/drive/MyDrive/Cross-Dataset Generalization Thesis/web app/model/"

# --- 1. DEFINE CLASS SETS ---
EUROSAT_CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

UCM_CLASSES = [
    'Agricultural', 'Airplane', 'BaseballDiamond', 'Beach', 'Buildings',
    'Chaparral', 'DenseResidential', 'Forest', 'Freeway', 'GolfCourse',
    'Harbor', 'Intersection', 'MediumResidential', 'MobileHomePark', 'Overpass',
    'ParkingLot', 'River', 'Runway', 'SparseResidential', 'StorageTanks', 'TennisCourt'
]

# --- 2. MODEL REGISTRY ---
MODEL_REGISTRY = {
    # EuroSAT Models
    "EuroSAT: MobileNetV2":    {"file": "eurosat_mobilenetv2_full.keras",    "classes": EUROSAT_CLASSES},
    "EuroSAT: ResNet50":       {"file": "eurosat_resnet50_full.keras",       "classes": EUROSAT_CLASSES},
    "EuroSAT: DenseNet121":    {"file": "eurosat_densenet121_full.keras",    "classes": EUROSAT_CLASSES},
    "EuroSAT: EfficientNetB0": {"file": "eurosat_efficientnetb0_full.keras", "classes": EUROSAT_CLASSES},

    # UC Merced Models
    "UCM: MobileNetV2":        {"file": "ucm_mobilenetv2_full.keras",        "classes": UCM_CLASSES},
    "UCM: ResNet50":           {"file": "ucm_resnet50_full.keras",           "classes": UCM_CLASSES},
    "UCM: DenseNet121":        {"file": "ucm_densenet121_full.keras",        "classes": UCM_CLASSES},
    "UCM: EfficientNetB0":     {"file": "ucm_efficientnetb0_full.keras",     "classes": UCM_CLASSES},
}

# --- 3. EXPANDED COLORS ---
LULC_COLORS = {
    'AnnualCrop': (255, 215, 0),       'Forest': (34, 139, 34),
    'HerbaceousVegetation': (154, 205, 50), 'Highway': (50, 50, 50),
    'Industrial': (220, 20, 60),       'Pasture': (124, 252, 0),
    'PermanentCrop': (139, 69, 19),    'Residential': (169, 169, 169),
    'River': (0, 191, 255),            'SeaLake': (0, 0, 139),
    'Agricultural': (218, 165, 32),    'Airplane': (230, 230, 250),
    'BaseballDiamond': (139, 69, 19),  'Beach': (244, 164, 96),
    'Buildings': (176, 196, 222),      'Chaparral': (85, 107, 47),
    'DenseResidential': (112, 128, 144),'Freeway': (47, 79, 79),
    'GolfCourse': (0, 100, 0),         'Harbor': (70, 130, 180),
    'Intersection': (105, 105, 105),   'MediumResidential': (119, 136, 153),
    'MobileHomePark': (210, 180, 140), 'Overpass': (128, 128, 128),
    'ParkingLot': (192, 192, 192),     'Runway': (220, 220, 220),
    'SparseResidential': (143, 188, 143),'StorageTanks': (255, 69, 0),
    'TennisCourt': (0, 255, 127)
}

# --- 4. METRICS FOR PLOTTING ---
# Note: UCM Accuracies are placeholders. Update these with your real training results.
MODEL_METRICS = {
    "EuroSAT: MobileNetV2":    {"Accuracy": "85.50%", "Params": "2.3M", "Desc": "Lightweight CNN"},
    "EuroSAT: ResNet50":       {"Accuracy": "98.50%", "Params": "25.6M", "Desc": "Residual Learning"},
    "EuroSAT: DenseNet121":    {"Accuracy": "98.17%", "Params": "8.0M", "Desc": "Densely Connected"},
    "EuroSAT: EfficientNetB0": {"Accuracy": "97.56%", "Params": "5.3M", "Desc": "Compound Scaling"},

    "UCM: MobileNetV2":        {"Accuracy": "91.67%", "Params": "2.3M", "Desc": "Lightweight CNN"},
    "UCM: ResNet50":           {"Accuracy": "94.76%", "Params": "25.6M", "Desc": "Residual Learning"},
    "UCM: DenseNet121":        {"Accuracy": "94.52%", "Params": "8.0M", "Desc": "Densely Connected"},
    "UCM: EfficientNetB0":     {"Accuracy": "94.05%", "Params": "5.3M", "Desc": "Compound Scaling"},
}

In [6]:
print("\n🔄 SYSTEM STARTUP SEQUENCE...")
print("—" * 60)

loaded_models = {}
model_input_shapes = {}
load_times = {}
load_status = {}

# Iterate through the Registry
for display_name, config in MODEL_REGISTRY.items():
    filename = config["file"]
    path = os.path.join(BASE_FOLDER, filename)
    load_status[display_name] = "⚠️ NOT FOUND"

    if os.path.exists(path):
        try:
            start_time = time.time()
            # Compile=False is safer for inference
            model = tf.keras.models.load_model(path, compile=False)
            load_time = (time.time() - start_time) * 1000

            loaded_models[display_name] = model
            load_times[display_name] = load_time

            # Get input shape safely
            try:
                cfg = model.input_shape
                if isinstance(cfg, list): cfg = cfg[0]
                # Store as (Width, Height)
                model_input_shapes[display_name] = (cfg[2], cfg[1]) if (cfg and len(cfg) >= 3) else (64, 64)
            except Exception:
                model_input_shapes[display_name] = (64, 64)

            load_status[display_name] = f"✅ ONLINE ({load_time:.0f}ms)"
            print(f"   {load_status[display_name]:<30} - {display_name}")

        except Exception as e:
            loaded_models[display_name] = None
            load_status[display_name] = f"❌ FAILED"
            print(f"   {load_status[display_name]:<30} - {display_name} ({str(e)[:20]})")
    else:
        print(f"   {load_status[display_name]:<30} - {display_name}")

print("—" * 60)
print(f"📊 Summary: {sum(1 for s in load_status.values() if 'ONLINE' in s)}/{len(MODEL_REGISTRY)} models loaded")

# Summary DataFrame
model_summary_df = pd.DataFrame({
    'Model Name': list(MODEL_REGISTRY.keys()),
    'Status': [load_status[m] for m in MODEL_REGISTRY.keys()],
    'Input Shape': [f"{model_input_shapes.get(m, (64,64))[0]}x{model_input_shapes.get(m, (64,64))[1]}" for m in MODEL_REGISTRY.keys()]
})


🔄 SYSTEM STARTUP SEQUENCE...
————————————————————————————————————————————————————————————
   ✅ ONLINE (1465ms)              - EuroSAT: MobileNetV2
   ✅ ONLINE (8884ms)              - EuroSAT: ResNet50
   ✅ ONLINE (8663ms)              - EuroSAT: DenseNet121
   ✅ ONLINE (6435ms)              - EuroSAT: EfficientNetB0
   ✅ ONLINE (4144ms)              - UCM: MobileNetV2
   ✅ ONLINE (10824ms)             - UCM: ResNet50
   ✅ ONLINE (7668ms)              - UCM: DenseNet121
   ✅ ONLINE (6022ms)              - UCM: EfficientNetB0
————————————————————————————————————————————————————————————
📊 Summary: 8/8 models loaded


In [7]:
def preprocess_tile(image, target_size):
    """Preprocess image for model input"""
    img = image.convert("RGB").resize(target_size)
    arr = tf.keras.preprocessing.image.img_to_array(img)
    arr = arr / 255.0
    return arr

def create_confidence_plot(confidences):
    """Create a bar plot of confidence scores"""
    if not confidences: return None
    colors = ['#10b981' if v > 0.8 else '#f59e0b' if v > 0.5 else '#ef4444' for v in confidences.values()]

    fig = go.Figure(data=[
        go.Bar(
            x=list(confidences.keys()),
            y=list(confidences.values()),
            marker_color=colors,
            text=[f'{v:.1%}' for v in confidences.values()],
            textposition='auto',
        )
    ])
    fig.update_layout(
        title="Confidence Scores by Class",
        xaxis_title="Land Cover Class",
        yaxis_title="Confidence",
        yaxis_range=[0, 1],
        template="plotly_dark",
        height=300,
        margin=dict(l=20, r=20, t=40, b=20)
    )
    return fig

# --- RESTORED FUNCTION ---
def generate_model_comparison_chart():
    """Generate comparison chart of all models"""
    models = list(MODEL_METRICS.keys())
    # Extract numerical values safely
    accuracies = [float(MODEL_METRICS[m]["Accuracy"].replace("%", "")) for m in models]
    params = [float(MODEL_METRICS[m]["Params"].replace("M", "")) for m in models]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Test Accuracy (%)", "Model Size (M Parameters)"),
        specs=[[{"type": "bar"}, {"type": "bar"}]]
    )

    # Color palette (EuroSAT=Greens/Blues, UCM=Oranges/Purples)
    colors = ['#34d399', '#3b82f6', '#8b5cf6', '#f59e0b', '#10b981', '#6366f1', '#a855f7', '#f97316']

    fig.add_trace(go.Bar(x=models, y=accuracies, marker_color=colors, name="Accuracy"), row=1, col=1)
    fig.add_trace(go.Bar(x=models, y=params, marker_color=colors, name="Parameters"), row=1, col=2)

    fig.update_layout(
        title="Model Performance Comparison (EuroSAT vs UCM)",
        template="plotly_dark",
        showlegend=False,
        height=400,
        margin=dict(l=20, r=20, t=50, b=50)
    )
    return fig

def get_legend_html(active_classes=None):
    """Generate HTML for Legend."""
    html = "<div class='legend-box'>"
    sorted_keys = sorted(LULC_COLORS.keys())
    for cls in sorted_keys:
        if active_classes and cls not in active_classes: continue
        col = LULC_COLORS[cls]
        hex_c = '#%02x%02x%02x' % col
        html += f"""
        <div class='legend-item'>
            <div style='width:16px;height:16px;background:{hex_c};border-radius:3px;border:1px solid rgba(255,255,255,0.2);'></div>
            <span style='font-size:0.8rem; color: #e2e8f0;'>{cls}</span>
        </div>
        """
    html += "</div>"
    return html

In [8]:
# --- ENGINE A: SINGLE IMAGE ANALYSIS ---
def analyze_single(image, model_display_name):
    if image is None: return None, "⚠️ Please upload an image first.", None, 0.0
    if not model_display_name: return None, "⚠️ Please select a model.", None, 0.0

    model = loaded_models.get(model_display_name)
    if model is None: return None, f"❌ Model not loaded.", None, 0.0

    try:
        # 1. Get Config for this specific model
        target = model_input_shapes.get(model_display_name, (64, 64))
        # VITAL: Get the correct class list (EuroSAT or UCM)
        current_classes = MODEL_REGISTRY[model_display_name]["classes"]

        # 2. Preprocess
        img_arr = np.expand_dims(preprocess_tile(image, target), axis=0)

        # 3. Predict
        start = time.time()
        pred = model.predict(img_arr, verbose=0)
        latency = (time.time() - start) * 1000

        pred_scores = pred[0]
        top_idx = np.argsort(pred_scores)[-5:][::-1]

        # 4. Map indices to class names using the CORRECT list
        results = {current_classes[i]: float(pred_scores[i]) for i in top_idx}
        top_class = current_classes[top_idx[0]]
        confidence = float(pred_scores[top_idx[0]])

        # 5. Log & HTML
        log = f"[{datetime.now().strftime('%H:%M:%S')}] MODEL: {model_display_name}\n" \
              f"[{datetime.now().strftime('%H:%M:%S')}] DATASET: {'EuroSAT' if 'EuroSAT' in model_display_name else 'UC Merced'}\n" \
              f"[{datetime.now().strftime('%H:%M:%S')}] PREDICTION: {top_class} ({confidence:.1%})"

        # HTML Card
        if confidence > 0.8: color, status = "#10b981", "HIGH CONFIDENCE"
        elif confidence > 0.5: color, status = "#f59e0b", "MODERATE"
        else: color, status = "#ef4444", "LOW CONFIDENCE"

        html = f"""
        <div class="result-card">
            <div style="text-align: center; padding: 20px;">
                <div style="font-size: 0.9rem; color: #94a3b8; margin-bottom: 10px;">PREDICTION RESULT</div>
                <div style="font-size: 2.0rem; font-weight: 800; color: white; margin-bottom: 5px;">{top_class.upper()}</div>
                <div style="font-size: 0.8rem; color: {color}; margin-bottom: 20px; letter-spacing: 2px;">{status}</div>
                <div style="display: inline-block; padding: 8px 20px; border-radius: 25px; background: {color}20; border: 2px solid {color};">
                    <span style="font-size: 1.2rem; font-weight: bold; color: {color};">{confidence:.1%}</span>
                </div>
                <div style="margin-top: 20px; font-size: 0.8rem; color: #cbd5e1;">
                    Model: {model_display_name} | Latency: {latency:.0f}ms
                </div>
            </div>
        </div>
        """

        fig = create_confidence_plot(results)
        return html, log, fig, confidence

    except Exception as e:
        return None, f"❌ Error: {str(e)}", None, 0.0


# --- ENGINE B: STRESS TEST ---
def run_stress_test(image, noise, blur):
    if image is None: return None, None, None, None

    # Apply distortions
    img = image.convert("RGB")
    if blur > 0: img = img.filter(ImageFilter.GaussianBlur(radius=blur))
    if noise > 0:
        arr = np.array(img, dtype=np.float32)
        noise_arr = np.random.normal(0, noise * 50, arr.shape)
        img = Image.fromarray(np.clip(arr + noise_arr, 0, 255).astype(np.uint8))

    results = []

    for name, model in loaded_models.items():
        if model:
            target = model_input_shapes.get(name, (64, 64))
            # Retrieve correct classes for this model
            current_classes = MODEL_REGISTRY[name]["classes"]

            img_arr = np.expand_dims(preprocess_tile(img, target), axis=0)

            start = time.time()
            pred = model.predict(img_arr, verbose=0)
            latency = (time.time() - start) * 1000

            idx = np.argmax(pred[0])
            confidence = np.max(pred[0])

            results.append({
                "Model": name,
                "Prediction": current_classes[idx], # Uses correct class list
                "Confidence": f"{confidence:.1%}",
                "Latency": f"{latency:.0f}ms"
            })

    df = pd.DataFrame(results).sort_values(by="Confidence", ascending=False)
    csv_path = "/content/stress_test_report.csv"
    df.to_csv(csv_path, index=False)

    # Simple comparison plot
    fig = go.Figure(data=[
        go.Bar(
            x=[r["Model"] for r in results],
            y=[float(r["Confidence"].replace("%", "")) / 100 for r in results],
            marker_color=['#10b981' if float(r["Confidence"].replace("%", "")) > 80 else '#ef4444' for r in results],
            text=[f"{r['Prediction']}<br>{r['Confidence']}" for r in results], # Show class name on bar
            textposition='auto',
        )
    ])
    fig.update_layout(title="Robustness Comparison", template="plotly_dark", height=400)

    return img, df, csv_path, fig


# --- ENGINE C: LULC MAPPING ---
def generate_lulc_side_by_side(large_image, model_display_name):
    if large_image is None or not model_display_name: return None, "⚠️ Missing input", None

    model = loaded_models.get(model_display_name)
    if model is None: return None, f"❌ Model not loaded", None

    # Get correct classes
    current_classes = MODEL_REGISTRY[model_display_name]["classes"]

    patch_size = 64
    w, h = large_image.size

    # Resize to grid multiple
    new_w = (w // patch_size) * patch_size
    new_h = (h // patch_size) * patch_size
    large_image = large_image.resize((new_w, new_h))

    map_canvas = Image.new('RGB', (new_w, new_h), (30, 30, 30))
    draw = ImageDraw.Draw(map_canvas)

    patches, coords = [], []
    for y in range(0, new_h, patch_size):
        for x in range(0, new_w, patch_size):
            box = (x, y, x + patch_size, y + patch_size)
            patch = large_image.crop(box)
            target = model_input_shapes.get(model_display_name, (64, 64))
            patches.append(preprocess_tile(patch, target))
            coords.append((x, y))

    if not patches: return large_image, "⚠️ No patches generated", None

    # Batch predict
    patches_arr = np.array(patches)
    preds = model.predict(patches_arr, verbose=0)
    indices = np.argmax(preds, axis=1)

    counts = {} # Dynamic counting

    for i, (x, y) in enumerate(coords):
        cls_name = current_classes[indices[i]]

        # Color lookup (safe)
        color = LULC_COLORS.get(cls_name, (128, 128, 128))

        draw.rectangle([x, y, x + patch_size, y + patch_size], fill=color)
        counts[cls_name] = counts.get(cls_name, 0) + 1

    # Combine images
    combined = Image.new('RGB', (new_w * 2, new_h))
    combined.paste(large_image, (0, 0))
    combined.paste(map_canvas, (new_w, 0))

    # Stats HTML
    total = sum(counts.values())
    stats = "<h4>🌍 Class Distribution</h4>"
    for cls, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
        if count > 0:
            pct = (count / total) * 100
            hex_c = '#%02x%02x%02x' % LULC_COLORS.get(cls, (128,128,128))
            stats += f"""
            <div style='margin:4px 0; padding:4px 8px; background:rgba(255,255,255,0.05); border-radius:4px; display:flex; align-items:center;'>
                <span style='width:10px; height:10px; background:{hex_c}; margin-right:8px; border-radius:2px;'></span>
                <span style='flex-grow:1; font-size:0.85rem;'>{cls}</span>
                <span style='font-weight:bold; font-size:0.85rem;'>{pct:.1f}%</span>
            </div>
            """

    # Pie Chart
    labels = list(counts.keys())
    values = list(counts.values())
    colors = [f"rgb{LULC_COLORS.get(k, (128,128,128))}" for k in labels]

    fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.4, marker_colors=colors)])
    fig.update_layout(title="Coverage Distribution", template="plotly_dark", height=300, margin=dict(t=40,b=20,l=20,r=20))

    return combined, stats, fig

In [ ]:
# --- FINAL UI CELL: ENHANCED VISUALS (STABLE + SAFE) ---
import gradio as gr

# ==========================================
# 1. COMPATIBILITY & SAFETY LAYER (CRITICAL)
# ==========================================
if "MODEL_REGISTRY" in globals() and isinstance(MODEL_REGISTRY, dict) and len(MODEL_REGISTRY) > 0:
    UI_MODEL_LIST = list(MODEL_REGISTRY.keys())
else:
    # Absolute-safe fallback (prevents invisible dropdown)
    UI_MODEL_LIST = ["EuroSAT: ResNet50", "UCM: ResNet50"]

# Legend helper (safe)
def get_legend_html_ui(active_classes=None):
    if "LULC_COLORS" not in globals():
        return "<div style='color:white'>Legend unavailable</div>"

    html = "<div class='legend-box'>"
    for cls in sorted(LULC_COLORS.keys()):
        if active_classes and cls not in active_classes:
            continue
        col = LULC_COLORS[cls]
        hex_c = '#%02x%02x%02x' % tuple(col)
        html += f"""
        <div class='legend-item'>
            <div style='width:12px; height:12px; background:{hex_c};
                        border-radius:3px; box-shadow:0 0 5px {hex_c};'></div>
            <span style='font-size:0.85rem; color:#cbd5e1; font-weight:500;'>{cls}</span>
        </div>
        """
    html += "</div>"
    return html

# ==========================================
# 2. VISUAL STYLING (UNCHANGED – SAFE)
# ==========================================
css = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@300;400;600&family=Inter:wght@300;400;600;700;800&display=swap');

/* Base styles with glassmorphism */
body, .gradio-container {
    background: linear-gradient(135deg, #0a0a2a 0%, #1a1b3a 30%, #2d1b69 100%) !important;
    font-family: 'Inter', sans-serif !important;
    color: #f0f4ff;
}

/* Glassmorphism containers */
.gradio-container .tabs,
.gradio-container .block,
.gradio-container .form,
.gradio-container .panel {
    background: rgba(20, 22, 40, 0.7) !important;
    backdrop-filter: blur(20px) saturate(180%);
    -webkit-backdrop-filter: blur(20px) saturate(180%);
    border: 1px solid rgba(255, 255, 255, 0.15);
    border-radius: 16px;
    box-shadow: 0 8px 32px rgba(0, 0, 0, 0.2);
}

/* Hero section with animated gradient */
.hero-section {
    text-align: center;
    padding: 40px 30px;
    background: rgba(15, 23, 42, 0.6);
    backdrop-filter: blur(20px);
    -webkit-backdrop-filter: blur(20px);
    border-bottom: 1px solid rgba(255, 255, 255, 0.1);
    border-radius: 0 0 24px 24px;
    margin-bottom: 32px;
    position: relative;
    overflow: hidden;
}

.hero-section::before {
    content: '';
    position: absolute;
    top: -50%;
    left: -50%;
    width: 200%;
    height: 200%;
    background: linear-gradient(
        45deg,
        transparent 30%,
        rgba(59, 130, 246, 0.1) 50%,
        transparent 70%
    );
    animation: shimmer 8s infinite linear;
    z-index: -1;
}

@keyframes shimmer {
    0% { transform: rotate(0deg); }
    100% { transform: rotate(360deg); }
}

/* Animated gradient title */
.hero-title {
    font-size: 3rem;
    font-weight: 900;
    background: linear-gradient(90deg,
        #34d399 0%,
        #3b82f6 25%,
        #8b5cf6 50%,
        #f59e0b 75%,
        #34d399 100%);
    background-size: 200% auto;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    animation: gradient 3s ease-in-out infinite;
    margin-bottom: 1rem;
    text-shadow: 0 0 30px rgba(59, 130, 246, 0.4),
                 0 0 60px rgba(139, 92, 246, 0.3),
                 0 0 90px rgba(52, 211, 153, 0.2);
}

@keyframes gradient {
    0% { background-position: 0% center; }
    50% { background-position: 100% center; }
    100% { background-position: 0% center; }
}

.hero-subtitle {
    font-size: 1.2rem;
    color: #c7d2fe;
    font-weight: 400;
    margin-bottom: 0.5rem;
    opacity: 0.9;
}

.hero-description {
    font-size: 0.95rem;
    color: #94a3b8;
    max-width: 600px;
    margin: 0 auto;
    line-height: 1.6;
}

/* Enhanced typography */
h1, h2, h3, h4, h5, h6 {
    font-family: 'Inter', sans-serif;
    font-weight: 700;
    color: #f0f4ff;
}

.gradio-container label {
    font-weight: 600 !important;
    color: #e2e8f0 !important;
    margin-bottom: 8px !important;
}

/* Modern cards with glassmorphism */
.result-card {
    background: rgba(30, 41, 59, 0.5);
    backdrop-filter: blur(15px);
    -webkit-backdrop-filter: blur(15px);
    border: 1px solid rgba(255, 255, 255, 0.15);
    border-radius: 16px;
    margin-top: 16px;
    transition: all 0.3s ease;
    padding: 24px;
    text-align: center;
}

.result-card:hover {
    transform: translateY(-2px);
    box-shadow: 0 12px 40px rgba(0, 0, 0, 0.3);
    border-color: rgba(59, 130, 246, 0.3);
}

/* Enhanced legend */
.legend-box {
    display: flex;
    flex-wrap: wrap;
    gap: 12px;
    padding: 20px;
    background: rgba(15, 23, 42, 0.6);
    backdrop-filter: blur(10px);
    border: 1px solid rgba(255, 255, 255, 0.1);
    border-radius: 12px;
    margin-top: 16px;
    max-height: 300px;
    overflow-y: auto;
}

.legend-item {
    display: flex;
    align-items: center;
    gap: 8px;
    background: rgba(255,255,255,0.05);
    padding: 8px 12px;
    border-radius: 8px;
    transition: all 0.2s ease;
}

.legend-item:hover {
    background: rgba(255,255,255,0.1);
    transform: translateX(4px);
}

/* Enhanced buttons with hover animations */
button {
    border-radius: 12px !important;
    padding: 12px 28px !important;
    font-weight: 600 !important;
    transition: all 0.3s ease !important;
    position: relative;
    overflow: hidden;
    border: none !important;
}

button:hover {
    transform: translateY(-4px) !important;
    box-shadow: 0 12px 30px rgba(59, 130, 246, 0.4) !important;
}

button:active {
    transform: translateY(-2px) !important;
}

/* Primary button glow effect */
button.primary {
    background: linear-gradient(135deg, #3b82f6 0%, #8b5cf6 100%) !important;
    box-shadow: 0 4px 20px rgba(59, 130, 246, 0.3) !important;
}

button.primary:hover {
    box-shadow: 0 8px 30px rgba(59, 130, 246, 0.6),
                0 0 0 3px rgba(59, 130, 246, 0.1) !important;
}

/* Stop button styling */
button.stop {
    background: linear-gradient(135deg, #ef4444 0%, #f59e0b 100%) !important;
}

/* Tabs styling */
.tabs {
    background: rgba(15, 23, 42, 0.7) !important;
    backdrop-filter: blur(15px);
    border-radius: 16px !important;
    padding: 20px !important;
}

.tab-nav {
    background: rgba(30, 41, 59, 0.8) !important;
    border-radius: 12px !important;
    padding: 8px !important;
    margin-bottom: 24px !important;
}

.tab-button {
    border-radius: 10px !important;
    padding: 12px 24px !important;
    font-weight: 600 !important;
    transition: all 0.3s ease !important;
}

.tab-button.selected {
    background: linear-gradient(135deg, #3b82f6 0%, #8b5cf6 100%) !important;
    box-shadow: 0 4px 15px rgba(59, 130, 246, 0.3) !important;
}

/* Input fields and dropdowns */
input, select, .gradio-dropdown, textarea {
    background: rgba(30, 41, 59, 0.7) !important;
    border: 1px solid rgba(255, 255, 255, 0.15) !important;
    border-radius: 12px !important;
    padding: 12px 16px !important;
    color: #f0f4ff !important;
    transition: all 0.3s ease !important;
}

input:focus, select:focus, .gradio-dropdown:focus {
    border-color: #3b82f6 !important;
    box-shadow: 0 0 0 3px rgba(59, 130, 246, 0.1) !important;
    outline: none !important;
}

/* Image containers */
.gr-image {
    border-radius: 16px !important;
    overflow: hidden;
    border: 1px solid rgba(255, 255, 255, 0.1);
}

/* Sliders */
.gr-slider {
    border-radius: 12px !important;
}

.gr-slider .track {
    background: rgba(59, 130, 246, 0.2) !important;
}

.gr-slider .track-fill {
    background: linear-gradient(90deg, #3b82f6, #8b5cf6) !important;
}

/* Dataframe styling */
.gr-dataframe {
    border-radius: 12px !important;
    overflow: hidden;
}

/* Plot containers - Fixed height to avoid squashing */
.plot-container {
    min-height: 500px !important;
    height: auto !important;
    overflow: visible !important;
    background: transparent !important;
}

/* 🔧 CRITICAL FIX: Allow dropdown popups to escape containers */
.gradio-container,
.gradio-container .block,
.gradio-container .form,
.gradio-container .panel,
.gradio-container .tabs {
    overflow: visible !important;
}
"""

# ==========================================
# 3. UI CONSTRUCTION
# ==========================================
with gr.Blocks(css=css, theme=gr.themes.Base()) as demo:

    # HERO
    gr.HTML("""
    <div class="hero-section">
        <div class="hero-title">🛰️ GeoVision AI: Multi-Domain</div>
        <div class="hero-subtitle">📡 Cross-Dataset Satellite Land Use Classification Platform</div>
        <div class="hero-description">
            Leveraging state-of-the-art deep learning models for EuroSAT & UC Merced datasets.
            Compare, analyze, and visualize LULC classifications with interactive tools.
        </div>
    </div>
    """)

    with gr.Tabs():

        # --------------------------------------------------
        # TAB 1: SINGLE IMAGE ANALYSIS
        # --------------------------------------------------
        with gr.TabItem("🔍 Single Image Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📤 Upload & Configuration")
                    input_img = gr.Image(label="📷 Upload Satellite Tile", type="pil", height=320)

                    model_selector = gr.Dropdown(
                        choices=UI_MODEL_LIST,
                        value=UI_MODEL_LIST[0],
                        label="🤖 Select Model Architecture",
                        interactive=True
                    )

                    btn_analyze = gr.Button("🚀 Analyze Image", variant="primary", size="lg")

                with gr.Column(scale=1):
                    gr.Markdown("### 📊 Results & Insights")
                    out_html = gr.HTML()
                    out_plot = gr.Plot()
                    out_log = gr.Textbox(max_lines=8, label="Logs")

            btn_analyze.click(
                fn=analyze_single,
                inputs=[input_img, model_selector],
                outputs=[out_html, out_log, out_plot, gr.Number(visible=False)]
            )

        # --------------------------------------------------
        # TAB 2: ROBUSTNESS TEST
        # --------------------------------------------------
        with gr.TabItem("⚡ Robustness Benchmark"):
            with gr.Row():
                with gr.Column(scale=1):
                    stress_img = gr.Image(type="pil", label="Base Image", height=250)
                    noise = gr.Slider(0, 1.0, value=0, step=0.1, label="Noise")
                    blur = gr.Slider(0, 5.0, value=0, step=0.5, label="Blur")
                    btn_stress = gr.Button("🔥 Run Benchmark", variant="stop")

                with gr.Column(scale=2):
                    out_distorted = gr.Image(height=220)
                    out_stress_plot = gr.Plot()
                    out_csv = gr.File()

            btn_stress.click(
                fn=run_stress_test,
                inputs=[stress_img, noise, blur],
                outputs=[out_distorted, gr.DataFrame(visible=False), out_csv, out_stress_plot]
            )

        # --------------------------------------------------
        # TAB 3: LULC MAPPING
        # --------------------------------------------------
        with gr.TabItem("🌍 LULC Mapping"):
            with gr.Row():
                with gr.Column(scale=1):
                    map_img = gr.Image(type="pil", label="Large Satellite Image")
                    map_model = gr.Dropdown(
                        choices=UI_MODEL_LIST,
                        value=UI_MODEL_LIST[0],
                        label="Classification Model",
                        interactive=True
                    )
                    btn_map = gr.Button("🗺️ Generate Map", variant="primary")

                    with gr.Accordion("🎨 Class Legend", open=False):
                        gr.HTML(get_legend_html_ui())

                with gr.Column(scale=2):
                    out_map = gr.Image()

                    # Stack these vertically instead of side-by-side
                    gr.Markdown("#### 📊 Distribution Analysis")
                    out_map_stats = gr.HTML()

                    # The plot now gets full width and the new 500px height from CSS
                    out_map_plot = gr.Plot(label="Coverage Distribution")

            btn_map.click(
                fn=generate_lulc_side_by_side,
                inputs=[map_img, map_model],
                outputs=[out_map, out_map_stats, out_map_plot]
            )

        # --------------------------------------------------
        # TAB 4: SYSTEM DASHBOARD
        # --------------------------------------------------
        with gr.TabItem("📊 System Dashboard"):
            if "model_summary_df" in globals():
                gr.DataFrame(model_summary_df)
            if "generate_model_comparison_chart" in globals():
                gr.Plot(generate_model_comparison_chart())

if __name__ == "__main__":
    demo.launch(debug=True, share=True)


/tmp/ipykernel_5282/2772317406.py:327: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Base()) as demo:
/tmp/ipykernel_5282/2772317406.py:327: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Base()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a365366b274b394490.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
